# Microwells — Position List Tools
1. Generate a full position grid from 3 marker wells
2. Merge two position lists with an optional Z offset

In [ ]:
import json, copy
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

XY_STAGE = 'XYStage:XY:31'
Z_STAGE  = 'ZStage:Z:32'

def make_position(label, x, y, z, grid_col=0, grid_row=0):
    """Build one position entry in the MM2.0 Property Map format."""
    return {
        'DefaultXYStage': {'type': 'STRING', 'scalar': XY_STAGE},
        'DefaultZStage':  {'type': 'STRING', 'scalar': Z_STAGE},
        'DevicePositions': {
            'type': 'PROPERTY_MAP',
            'array': [
                {'Device': {'type': 'STRING', 'scalar': Z_STAGE},
                 'Position_um': {'type': 'DOUBLE', 'array': [float(z)]}},
                {'Device': {'type': 'STRING', 'scalar': XY_STAGE},
                 'Position_um': {'type': 'DOUBLE', 'array': [float(x), float(y)]}},
            ],
        },
        'GridCol':    {'type': 'INTEGER',      'scalar': grid_col},
        'GridRow':    {'type': 'INTEGER',      'scalar': grid_row},
        'Label':      {'type': 'STRING',       'scalar': label},
        'Properties': {'type': 'PROPERTY_MAP', 'scalar': {}},
    }

def save_pos_file(path, positions):
    data = {
        'encoding': 'UTF-8',
        'format': 'Micro-Manager Property Map',
        'major_version': 2,
        'minor_version': 0,
        'map': {'StagePositions': {'type': 'PROPERTY_MAP', 'array': positions}},
    }
    with open(path, 'w') as f:
        json.dump(data, f, indent=2)

def load_pos_file(path):
    """Load positions from a MM2.0 Property Map .pos file."""
    with open(path) as f:
        data = json.load(f)
    return data['map']['StagePositions']['array']

def read_xyz_old(pos):
    """Read X, Y, Z from an old-format (VERSION 3) position entry."""
    return np.array([
        pos['DEVICES'][2]['X'],
        pos['DEVICES'][2]['Y'],
        pos['DEVICES'][0]['X'],
    ], dtype=float)

print('Helpers loaded.')

---
## 1. Generate position list from marker wells

Requires a `.pos` file with exactly **3 positions**:
- `Pos0` — origin (top-left well)
- `Pos1` — end of the first row (top-right well)
- `Pos2` — start of the second row (well below origin)

### Settings

In [ ]:
FOLDER   = r'E:/UserData/Nazar/20260626/'
FILE_IN  = 'PositionList_W1.pos'     # 3-marker input (old MM format)
FILE_OUT = 'PositionList_W1_full.pos'

# --- Plate size (choose one or set custom) ---
# Small plate:  23 × 26 wells
# LENGTH, WIDTH = 22, 25

# Large plate:  35 × 40 wells
LENGTH, WIDTH = 34, 39   # steps between wells (wells = steps + 1)

### Generate and save

In [ ]:
# Marker file is in old VERSION 3 format (created manually in MM)
with open(FOLDER + FILE_IN) as f:
    raw = json.load(f)

origin     = read_xyz_old(raw['POSITIONS'][0])
position_1 = read_xyz_old(raw['POSITIONS'][1])
position_2 = read_xyz_old(raw['POSITIONS'][2])

vector_1 = (position_1 - origin) / LENGTH
vector_2 = (position_2 - origin) / LENGTH

print(f'Vector 1 (row step) : {vector_1}')
print(f'Vector 2 (col step) : {vector_2}')

current   = origin.copy()
direction = True
label     = 0
positions = []
xs, ys    = [], []

for ii in range(WIDTH):
    step = vector_1 if direction else -vector_1
    for jj in range(LENGTH):
        positions.append(make_position(f'Pos{label}', current[0], current[1], current[2]))
        xs.append(current[0]); ys.append(current[1])
        current += step
        label   += 1
    direction = not direction
    current  += vector_2 if direction else vector_2 - vector_1

save_pos_file(FOLDER + FILE_OUT, positions)
print(f'Generated {label} positions  ->  {FOLDER + FILE_OUT}')

### Plot

In [ ]:
plt.figure(figsize=(10, 8))
plt.scatter(xs, ys, marker='.', s=10, label='Grid')
plt.scatter(*origin[:2],     marker='*', s=200, zorder=5, label='Origin')
plt.scatter(*position_1[:2], marker='*', s=200, zorder=5, label='Position 1')
plt.scatter(*position_2[:2], marker='*', s=200, zorder=5, label='Position 2')
plt.xlabel('X (um)'); plt.ylabel('Y (um)')
plt.title(f'{LENGTH+1} x {WIDTH} wells  —  {label} total positions')
plt.legend(); plt.axis('equal'); plt.grid(True)
plt.tight_layout(); plt.show()

---
## 2. Merge two position lists

### Settings

In [ ]:
MERGE_FOLDER   = r'E:/UserData/Nazar/20260626/'
MERGE_FILE_1   = 'PositionList_W1_full.pos'
MERGE_FILE_2   = 'PositionList_W2_full.pos'
MERGE_FILE_OUT = 'PositionList_merged.pos'

Z_OFFSET_UM = 40   # µm to add to every Z position (set to 0 to skip)

### Merge and save

In [ ]:
pos1 = load_pos_file(MERGE_FOLDER + MERGE_FILE_1)
pos2 = load_pos_file(MERGE_FOLDER + MERGE_FILE_2)
n1, n2 = len(pos1), len(pos2)

# Re-label pos2 continuing from end of pos1
for p in pos2:
    idx = int(p['Label']['scalar'].replace('Pos', '')) + n1
    p['Label']['scalar'] = f'Pos{idx}'

merged = pos1 + pos2

# Apply Z offset
if Z_OFFSET_UM:
    for p in merged:
        for dev in p['DevicePositions']['array']:
            if dev['Device']['scalar'] == Z_STAGE:
                dev['Position_um']['array'][0] += Z_OFFSET_UM

save_pos_file(MERGE_FOLDER + MERGE_FILE_OUT, merged)
print(f'File 1   : {n1} positions')
print(f'File 2   : {n2} positions')
print(f'Total    : {n1 + n2} positions')
print(f'Z offset : +{Z_OFFSET_UM} um')
print(f'Saved to : {MERGE_FOLDER + MERGE_FILE_OUT}')

### Plot

In [ ]:
def get_xy(positions):
    xs, ys = [], []
    for p in positions:
        for dev in p['DevicePositions']['array']:
            if dev['Device']['scalar'] == XY_STAGE:
                xs.append(dev['Position_um']['array'][0])
                ys.append(dev['Position_um']['array'][1])
    return xs, ys

xs1, ys1 = get_xy(merged[:n1])
xs2, ys2 = get_xy(merged[n1:])

plt.figure(figsize=(10, 8))
plt.scatter(xs1, ys1, c='steelblue', marker='.', s=10)
plt.scatter(xs2, ys2, c='tomato',    marker='.', s=10)
plt.legend(handles=[
    mpatches.Patch(color='steelblue', label=f'{MERGE_FILE_1}  ({n1} pos)'),
    mpatches.Patch(color='tomato',    label=f'{MERGE_FILE_2}  ({n2} pos)'),
])
plt.xlabel('X (um)'); plt.ylabel('Y (um)')
plt.title(f'Merged — {n1 + n2} total positions')
plt.axis('equal'); plt.grid(True)
plt.tight_layout(); plt.show()